# 00 · Kiểm tra môi trường Kaggle và chốt requirements.txt (ticket #2)

Chắc chắn implicit (ALS), gensim (item2vec) và PyTorch (SASRec, mix học) chạy được trên image Kaggle hiện tại trước khi viết mô hình (#14–#16, #23). Notebook cũng chạy bộ test của repo để kiểm tra `src/` trên Python và các gói của Kaggle.

**Input:** không cần dataset. Bật **Internet** (để `pip install` và clone repo). Chỉ cần CPU.

Chạy **Run All**, rồi chép khối `requirements.txt` ở ô cuối và các dòng `LỖI` (nếu có) về cho nhóm.

In [ ]:
import importlib, importlib.metadata as md, platform, sys

PACKAGES = {  # tên trên pip → tên module để import
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "scipy": "scipy",
    "implicit": "implicit", "gensim": "gensim", "torch": "torch", "pytest": "pytest",
}

def installed() -> dict[str, str | None]:
    out = {}
    for pip_name in PACKAGES:
        try:
            out[pip_name] = md.version(pip_name)
        except md.PackageNotFoundError:
            out[pip_name] = None
    return out

print("Python", sys.version.split()[0], "|", platform.platform())
before = installed()
for name, version in before.items():
    print(f"{name:10s} {version or 'CHƯA CÓ'}")

In [ ]:
# Cài những gói còn thiếu (thường là implicit). Nếu ô này báo lỗi, chép nguyên log về: đó là thứ cần ghi vào README.
missing = [name for name, version in before.items() if version is None]
print("cài thêm:", missing or "không có")
if missing:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)  # lỗi thì dừng ở đây
importlib.invalidate_caches()

In [ ]:
# Bootstrap repo và chạy bộ test: src/ phải chạy được trên Python + numpy/pandas của Kaggle.
BRANCH = "main"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
!cd /kaggle/working/MusicRecSys && python -m pytest -q 2>&1 | tail -3

In [ ]:
# ALS (implicit) trên ma trận playlist × bài đồ chơi, ép chạy CPU
import numpy as np
import scipy.sparse as sp
from implicit.cpu.als import AlternatingLeastSquares

rng = np.random.default_rng(0)
user_items = sp.random(300, 120, density=0.05, format="csr", random_state=0, dtype=np.float32)
user_items.data[:] = 1.0
als = AlternatingLeastSquares(factors=16, iterations=5, regularization=0.01, random_state=0)
als.fit(user_items, show_progress=False)
ids, scores = als.recommend(0, user_items[0], N=5)
print("ALS OK | item_factors", als.item_factors.shape, "| top-5 của playlist 0:", ids.tolist())

In [ ]:
# item2vec: Word2Vec (gensim) trên chuỗi bài của từng playlist đồ chơi
from gensim.models import Word2Vec

playlists = [[f"t{int(i)}" for i in rng.integers(0, 120, 20)] for _ in range(300)]
w2v = Word2Vec(playlists, vector_size=16, window=5, min_count=1, sg=1, epochs=5, workers=1, seed=0)
print("Word2Vec OK | số bài", len(w2v.wv), "| gần t0 nhất:", [t for t, _ in w2v.wv.most_similar("t0", topn=3)])

In [ ]:
# PyTorch: vài bước train một mô hình nhỏ (embedding + linear, Adam), loss phải giảm
import torch

torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Embedding(120, 16), torch.nn.Flatten(), torch.nn.Linear(16 * 5, 120))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
x, y = torch.randint(0, 120, (256, 5)), torch.randint(0, 120, (256,))
losses = []
for _ in range(20):
    optimizer.zero_grad()
    loss = torch.nn.functional.cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print(f"PyTorch OK | loss {losses[0]:.3f} → {losses[-1]:.3f} | CUDA: {torch.cuda.is_available()}")
assert losses[-1] < losses[0], "LỖI: loss không giảm" 

In [ ]:
# Khối requirements.txt để chép về repo. Bỏ phần +cu… của torch vì PyPI không có bản đó.
from datetime import date

after = installed()
print(f"# Đã thử trên Kaggle {date.today():%d/%m/%Y}, Python {sys.version.split()[0]} (notebooks/00_env_check.ipynb)")
for name, version in after.items():
    print(f"{name}=={version.split('+')[0]}" if version else f"# LỖI: {name} không cài được")

## Kết quả (chạy trên Kaggle, CPU, 25/11/2026)

- **Python** 3.13.18, Linux x86_64 (glibc 2.39).
- **Có sẵn:** numpy 2.1.3, pandas 2.3.3, pyarrow 23.0.1, scipy 1.16.3, gensim 4.4.0, torch 2.11.0+cpu, pytest 8.4.2.
- **Phải cài thêm:** implicit 0.7.3 (`pip install`, cần bật Internet); cài được ngay, không cần vá.
- **Bộ test của repo:** 194 passed in 4.38s trên `main`.
- **ALS, Word2Vec, một bước train PyTorch:** chạy hết, không lỗi (Run All dừng ở ô lỗi đầu tiên, và ô cuối đã chạy).
- `requirements.txt` ở gốc repo chép từ ô trên.